# Fine-tune Mamba-130M to be Yo-bot 🐍

Fine-tunes [`state-spaces/mamba-130m-hf`](https://huggingface.co/state-spaces/mamba-130m-hf) on Q&A pairs about Chaiyo (`dataset.jsonl`).

**Cost: $0** — run on a free Colab T4 GPU (`Runtime → Change runtime type → T4 GPU`). Training takes ~5–10 minutes.

Steps: install → load model → load dataset → train → test → push to Hugging Face Hub → deploy the Space in `mamba/hf-space/`.

In [ ]:
%pip install -q -U transformers datasets accelerate
# Optional speedup (CUDA kernels). Fine-tuning works without them (slower):
# %pip install -q mamba-ssm causal-conv1d

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'state-spaces/mamba-130m-hf'
device = 'cuda' if torch.cuda.is_available() else 'cpu'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(device)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.0f}M params on {device}')

In [ ]:
# Upload dataset.jsonl from the mamba/ folder of your portfolio repo
# (or fetch it from GitHub raw if the repo is public).
from google.colab import files
uploaded = files.upload()  # pick dataset.jsonl

from datasets import load_dataset
ds = load_dataset('json', data_files='dataset.jsonl', split='train')
print(ds)
print(ds[0]['text'])

In [ ]:
MAX_LEN = 256

def tokenize(batch):
    # Append EOS so the model learns where answers END (critical for a tiny model).
    texts = [t + tokenizer.eos_token for t in batch['text']]
    out = tokenizer(texts, truncation=True, max_length=MAX_LEN, padding='max_length')
    out['labels'] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in ids]
        for ids in out['input_ids']
    ]
    return out

tokenized = ds.map(tokenize, batched=True, remove_columns=ds.column_names)

In [ ]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir='yo-bot-mamba',
    num_train_epochs=15,          # small dataset → more epochs; watch loss
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy='no',
    fp16=torch.cuda.is_available(),
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized)
trainer.train()

In [ ]:
def ask(question, max_new_tokens=80):
    prompt = f'Q: {question}\nA:'
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        repetition_penalty=1.3,
        eos_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text.split('A:', 1)[-1].strip()

for q in ['Who is Chaiyo?', 'Where does Chaiyo work?', 'What is Gig&Co?', 'Are you ChatGPT?']:
    print('Q:', q)
    print('A:', ask(q))
    print()

In [ ]:
# Push to your Hugging Face account (free). Get a WRITE token at
# https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

REPO = 'yovatcha/yo-bot-mamba-130m'  # change to your username
model.push_to_hub(REPO)
tokenizer.push_to_hub(REPO)
print('Done → deploy mamba/hf-space/ as a Docker Space pointing at', REPO)

## Tips
- Answers loop or ramble? Raise `repetition_penalty`, lower `max_new_tokens`, or train fewer epochs.
- Answers wrong/mixed-up? Add more paraphrased Q&A variants to `dataset.jsonl` — tiny models need repetition in many phrasings.
- Loss near 0 quickly = memorization. That's expected (and fine) for this showcase; it's honest to label the mode "experimental".
- Want a from-scratch experiment later? Same notebook works — swap `from_pretrained` for `MambaForCausalLM(MambaConfig(...))`, but expect gibberish without huge data.